# Demo 1 · Fabric AI Functions sobre feedback de clientes (FraSoHome)

Enriquecer texto a escala con una sola línea de código: sentimiento, clasificación y extracción.

**Requisitos:** Fabric Runtime 1.3+ y un Lakehouse por defecto con los CSV en `Files/frasohome/`.
El despliegue (`scripts/deploy.ps1`) sube los datos y asocia este Lakehouse automáticamente.


## 1. Entorno


In [ ]:
import synapse.ml.aifunc as aifunc
import pandas as pd

BASE = "/lakehouse/default/Files/frasohome"  # Lakehouse por defecto asociado a este notebook


## 2. Cargar el feedback de clientes


In [ ]:
feedback = pd.read_csv(f"{BASE}/feedback_clientes.csv")
df = feedback[["texto_feedback", "sentimiento", "tema_principal", "rating_1_5"]].head(20).copy()
display(df)


## 3. Sentimiento y validación contra la etiqueta humana


In [ ]:
df["sentimiento_ai"] = df["texto_feedback"].ai.analyze_sentiment()
display(df[["texto_feedback", "sentimiento", "sentimiento_ai"]])


## 4. Clasificar el feedback por tema (categorías reales del negocio)


In [ ]:
df["tema_ai"] = df["texto_feedback"].ai.classify(
    "precio_promocion", "calidad_producto", "medidas_ajuste",
    "entrega_servicio", "tienda_experiencia"
)
display(df[["texto_feedback", "tema_principal", "tema_ai"]])


## 5. Inferir la categoría de producto desde el texto


In [ ]:
df["categoria_producto"] = df["texto_feedback"].ai.classify(
    "Sofás", "Sillas", "Mesas", "Dormitorio", "Iluminación",
    "Textil", "Pared", "Jardín"
)
display(df[["texto_feedback", "categoria_producto"]])


## 6. Extraer entidades del feedback


In [ ]:
entidades = df["texto_feedback"].ai.extract("producto", "atributo_criticado")
display(entidades)


In [ ]:
# Variante avanzada (novedad GA): ExtractLabel con esquema tipado
from synapse.ml.aifunc import ExtractLabel

# Para type="array" debemos especificar el esquema de los elementos mediante `properties`
# En este caso, cada elemento del array será un string.

df_ext = df["texto_feedback"].ai.extract(
    ExtractLabel(
        label="producto",
        description="Nombre del producto mencionado",
        type="string",
    ),
    ExtractLabel(
        label="problemas",
        description="Lista de problemas o quejas citadas",
        type="array",
        max_items=3,
        # Esquema de los elementos del array: cada item es un string
        properties={
            "items": {"type": "string"}
        },
    ),
)

display(df_ext)


## 7. Tickets de atención: resumen, clasificación y borrador de respuesta


In [ ]:
tickets = pd.read_csv(f"{BASE}/tickets_atencion_cliente.csv")
tk = tickets[["texto_cliente", "tipo_caso", "motivo_contacto", "csat_1_5"]].head(15).copy()

tk["resumen"] = tk["texto_cliente"].ai.summarize(
    instructions="Resume en una frase el problema del cliente, en español neutro."
)
tk["tipo_ai"] = tk["texto_cliente"].ai.classify(
    "devolucion", "info_producto", "entrega", "defecto",
    "stock_tienda", "cupon", "fidelizacion"
)
tk["borrador_respuesta"] = tk.ai.generate_response(
    "Redacta en 2 frases una respuesta empática y accionable al cliente, en español."
)
display(tk)


## 8. Coste y consumo


In [ ]:
# Tokens, modelo y filas OK/error de la última función
df["tema_ai"].ai.stats


> El consumo se mide en Capacity Units (CU) por tokens y aparece en la **Fabric Capacity Metrics app** como operación **AI Functions**.
> Prototipa con muestras (`.head(20)`) antes de procesar todo el histórico.
